# Homework set 3

Please **submit this Jupyter notebook through Canvas** no later than **Thursday November 19, 11:00**. **Submit the notebook file with your answers (as .ipynb file) and a pdf printout. The pdf version can be used by the teachers to provide feedback. On canvas there are hints about creating a nice pdf version.**

Before you hand in, please make sure the notebook runs, by running "Restart kernel and run all cells..." from the Kernel menu.

Homework is in **groups of two**, and you are expected to hand in original work. Work that is copied from another group will not be accepted.

# Exercise 0
Timothy Bohm: 17068061

Meric Ikiz: 13940155

Run the following cell to import NumPy, Matplotlib. If anything else is needed you can import this yourself.

In [1]:
import numpy as np
import matplotlib.pyplot as plt


# Exercise 1
N.B.1. you are to implement the methods yourself.

N.B.2. Tentative distribution of points is 2+1+2+2+2 points (plus 1 point makes 10).

Given a function $f$, let $T(f,a,b,m)$ denote the composite trapezoid rule with $m$ subintervals over the interval $[a,b]$. 

## (a)
Approximate the integral of $x^{-3}$ over $[a,b] = [ \frac{1}{10}, 100 ]$ by the composite trapezoid rule $T(f,a,b,m)$ for $m = 2^k$. Plot the absolute approximation error for different values of $k$. Find the smallest $k$ such that the exact error is less than $\epsilon = 10^{-3}$. Explain the slow convergence.

## (b)

To improve the convergence rate of the above problem, we may use an adaptive strategy, as discussed in the book and the lecture. Consider the following formulas for approximate integration
$$I_1(f,a,b) = T(f,a,b,1)$$
$$I_2(f,a,b) = T(f,a,b,2).$$
Show, based on the precise error estimates for the trapezoid rule from the book/lecture that the error in $I_2$ can be estimated by a formula of the form 
$$E_2 = C (I_1 - I_2)$$
and determine the constant $C$ (if you can't find $C$, you may take $C = 0.5$).

## (c)
An adaptive strategy for computing the integral on an interval $[a,b]$ now is: Compute $I_2$ and $E_2$, and accept $I_2$ as an approximation when the estimated error $E_2$ is less or equal than a desired tolerance $\epsilon$.  Otherwise, apply the procedure to 
$\int_a^{\frac{b+a}{2}} f(x) \, dx$ and $\int_{\frac{b+a}{2}}^b f(x) \, dx$ with tolerances $\frac{\epsilon}{2}$.

Write a recursive python routine that implements the adaptive strategy.

Then apply this routine to the function $x^{-3}$ with $a, b, \epsilon$ as before. What is the exact error in the obtained approximation? 

In [1]:
def adaptive_trapezoid(func, a, b, tolerance):
    midpoint = (a + b) / 2

    #Approximation with one subinterval
    I1 = (b - a) / 2 * (func(a) + func(b))

    #Approximation with two subintervals
    I2 = (b - a) / 4 * (func(a) + 2 * func(midpoint) + func(b))

    #Estimate the absolute error
    estimated_error = abs(I1 - I2) / 3

    if estimated_error <= tolerance:
        return I2

    #Split the interval and recursively call the function on each half
    left_integral = adaptive_trapezoid(func, a, midpoint, tolerance / 2)
    right_integral = adaptive_trapezoid(func, midpoint, b, tolerance / 2)

    return left_integral + right_integral

In [2]:
def func(x):
    return x**(-3)

a = 0.1
b = 100
tolerance = 1e-3

approximation = adaptive_trapezoid(func, a, b, tolerance)

exact_integral = 0.5 * (1 / a**2 - 1 / b**2)
actual_error = abs(exact_integral - approximation)

print("Approximation:", approximation)
print("Exact integral:", exact_integral)
print("Actual error:", actual_error)

Approximation: 50.00014849011892
Exact integral: 49.99994999999999
Actual error: 0.0001984901189260313


Results

Using the recursive adaptive trapezoid method with $C=\frac{1}{3}$ and tolerance $\epsilon=10^{-3}$, we obtain:

- Approximation: $50.00014849$
- Exact integral: $49.99995$
- Actual absolute error: $1.9849\times10^{-4}$

The actual error is smaller than the required tolerance of $10^{-3}$, so the method achieves the desired accuracy.

## (d)

Modify the code of (c) so that the number of function evaluations is counted. Optimize your implementation such that no unnecessary function evaluations are performed.

Compare the number of function evaluations used in the adaptive strategy of (c) with the result of (a). 
(*Hint*: To count the number of function evaluations, you may use a global variable.)


In [3]:
def adaptive_trapezoid_optimized(func, a, b, tolerance):
    function_evaluations = 0

    def evaluate(x):
        nonlocal function_evaluations
        function_evaluations += 1
        return func(x)

    def integrate(left, right, f_left, f_right, local_tolerance):
        midpoint = (left + right) / 2
        f_midpoint = evaluate(midpoint)

        #One trapezoid
        I1 = (right - left) / 2 * (f_left + f_right)

        #Two trapezoids
        I2 = (right - left) / 4 * (f_left + 2 * f_midpoint + f_right)

        estimated_error = abs(I1 - I2) / 3

        if estimated_error <= local_tolerance:
            return I2

        left_integral = integrate(left, midpoint, f_left, f_midpoint, local_tolerance / 2)
        right_integral = integrate(midpoint, right, f_midpoint, f_right, local_tolerance / 2)

        return left_integral + right_integral

    # Evaluate the original endpoints only once
    f_left = evaluate(a)
    f_right = evaluate(b)

    approximation = integrate(a, b, f_left, f_right, tolerance)

    return approximation, function_evaluations

In [4]:
def func(x):
    return x**(-3)

interval_start = 0.1
interval_end = 100
tolerance = 1e-3

approximation, evaluations = adaptive_trapezoid_optimized(
    func, interval_start, interval_end, tolerance
)

exact_integral = 0.5 * (1 / interval_start**2 - 1 / interval_end**2)
actual_error = abs(exact_integral - approximation)

print("Approximation:", approximation)
print("Exact integral:", exact_integral)
print("Actual error:", actual_error)
print("Function evaluations:", evaluations)

Approximation: 50.00014849011892
Exact integral: 49.99994999999999
Actual error: 0.0001984901189260313
Function evaluations: 9671


Results

The optimized adaptive trapezoid method reuses previously calculated function values by passing them to the recursive calls. This avoids evaluating the same points multiple times.

With tolerance $\epsilon=10^{-3}$, the method gives:

- Approximation: $50.00014849$
- Actual absolute error: $1.9849\times10^{-4}$
- Function evaluations: $9671$

Need part a. to do the comparison...

## (e)
In the course of executing the recursive procedure, some subintervals are refined (split into two subintervals) while others aren't as a result of the choices made by the algorithm. It turns out that the choices made by this algorithm are not always optimal. Other algorithms, that decide in a different way which subinterval needs to be refined, may be more efficient in the sense that they require less function evaluations (while using the same formulas for the approximate integral and the approximate error associated with a subinterval).

Can you explain why this is the case? Devise an alternative, non-recursive algorithm that addresses this issue and should to lead a more efficient integral computation. Describe your approach and algorithm in about 5 to 10 sentences (bullet points).
